In [8]:
!pip install gdown
import gdown

# File ID
teams_id = "1CKkXndq0dhs3jF2ST9DggH3cruuVC45R"
players_id = "1MFYOr41I3dDU6T6CmRS6zkNhIS8bHnAw"

# Output file
teams_file = "male_teams.csv"
players_file = "male_players.csv"

# Download (quan trọng: fuzzy=True)
gdown.download(id=teams_id, output=teams_file, fuzzy=True)
gdown.download(id=players_id, output=players_file, fuzzy=True)

Downloading...
From (original): https://drive.google.com/uc?id=1CKkXndq0dhs3jF2ST9DggH3cruuVC45R
From (redirected): https://drive.google.com/uc?id=1CKkXndq0dhs3jF2ST9DggH3cruuVC45R&confirm=t&uuid=27b934b9-6490-4c94-900d-87eeadc53930
To: /content/male_teams.csv
100%|██████████| 113M/113M [00:00<00:00, 176MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1MFYOr41I3dDU6T6CmRS6zkNhIS8bHnAw
From (redirected): https://drive.google.com/uc?id=1MFYOr41I3dDU6T6CmRS6zkNhIS8bHnAw&confirm=t&uuid=df81df66-7b01-4634-8816-11385be8d4ec
To: /content/male_players.csv
100%|██████████| 5.64G/5.64G [01:16<00:00, 73.8MB/s]


'male_players.csv'

##Load dữ liệu vào PySpark

In [9]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("FIFA23").getOrCreate()

teams_df = spark.read.csv(teams_file, header=True, inferSchema=True)
players_df = spark.read.csv(players_file, header=True, inferSchema=True)

##Kiểm tra dữ liệu đã load

In [10]:
teams_df.show(5)
players_df.show(5)

+-------+--------------------+------------+-----------+----------------+-------------------+---------+--------------------+------------+--------------+----------------+-------+------+--------+-------+--------+--------------------+----------+----------------------+-----------------+-------------------+--------------+-----------------------+----------------------+-------+---------------+--------------+--------------------+---------------------+---------+-----------+------------+--------------------+--------------+--------------+--------------------+----------------------+-----------------+-------------------------+---------+-----------------+-------------------+--------------+------------------+-----------+--------------+-------------------+-----------------------+---------------------+-------------------------+-----------------------+------------------------+------------------------+---------------------------+
|team_id|            team_url|fifa_version|fifa_update|fifa_update_date|    

##DATA CLEANING

###Kiểm tra dữ liệu thiếu (NULL, NaN)

In [11]:
# Import thư viện
from pyspark.sql.functions import col, isnan, when, count, sum as spark_sum, round

In [13]:
# Tạo version
## Version 1 (chỉ kiểm tra, không thay đổi dữ liệu)
players_v1 = players_df
teams_v1 = teams_df

In [22]:
# Kiểm tra NULL/NaN theo từng cột
from pyspark.sql.types import DoubleType, FloatType

def check_missing(df, name):
    print(f"\n===== MISSING VALUES: {name} =====")

    exprs = []

    for c, dtype in df.dtypes:
        # Nếu là số → check cả NaN + NULL
        if dtype in ["double", "float", "int", "bigint"]:
            exprs.append(
                count(when(col(c).isNull() | isnan(col(c)), c)).alias(c)
            )
        else:
            # Nếu không phải số → chỉ check NULL
            exprs.append(
                count(when(col(c).isNull(), c)).alias(c)
            )

    df.select(exprs).show(truncate=False)
check_missing(players_v1, "male_players")
check_missing(teams_v1, "male_teams")


===== MISSING VALUES: male_players =====
+---------+----------+------------+-----------+----------------+----------+---------+----------------+-------+---------+---------+--------+---+---+---------+---------+---------+-----------+------------+------------+---------+-------------+------------------+----------------+----------------+------------------------------+--------------+----------------+--------------+---------------+--------------------+--------------+---------+-----------+------------------------+---------+---------+---------+------------------+-----------+-------------+-------+--------+-------+---------+---------+-------+------------------+-------------------+--------------------------+-----------------------+-----------------+---------------+-----------+-----------------+------------------+------------------+---------------------+---------------------+----------------+------------------+----------------+----------------+-------------+-------------+--------------+------------

In [25]:
# Tổng số giá trị thiếu
def total_missing_fast(df, name):
    print(f"\n===== TOTAL MISSING: {name} =====")

    exprs = []

    for c, dtype in df.dtypes:
        # Nếu là số → check NULL + NaN
        if dtype in ["double", "float", "int", "bigint"]:
            exprs.append(
                count(when(col(c).isNull() | isnan(col(c)), c)).alias(c)
            )
        else:
            # Nếu không phải số → chỉ check NULL
            exprs.append(
                count(when(col(c).isNull(), c)).alias(c)
            )

    # Lấy kết quả về driver
    result = df.select(exprs).collect()[0]

    # Cộng tất cả giá trị lại
    total = sum(result)

    print("Total missing:", total)
total_missing_fast(players_v1, "male_players")
total_missing_fast(teams_v1, "male_teams")


===== TOTAL MISSING: male_players =====
Total missing: 75828244

===== TOTAL MISSING: male_teams =====
Total missing: 4172566


In [26]:
# Tỷ lệ % dữ liệu thiếu
def missing_percentage(df, name):
    print(f"\n===== MISSING PERCENTAGE: {name} =====")

    total_rows = df.count()
    exprs = []

    for c, dtype in df.dtypes:
        if dtype in ["double", "float", "int", "bigint"]:
            exprs.append(
                (count(when(col(c).isNull() | isnan(col(c)), c)) / total_rows * 100).alias(c)
            )
        else:
            exprs.append(
                (count(when(col(c).isNull(), c)) / total_rows * 100).alias(c)
            )

    df.select(exprs).show(truncate=False)
missing_percentage(players_v1, "male_players")
missing_percentage(teams_v1, "male_teams")


===== MISSING PERCENTAGE: male_players =====
+---------+----------+------------+-----------+----------------+----------+---------+----------------+-------+---------+------------------+-----------------+---+---+---------+---------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+-----------------+-----------------+------------------------------+--------------+----------------+-----------------+-----------------+--------------------+--------------+---------+-----------+------------------------+---------+---------+---------+------------------+-----------------+-----------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+-------------------+--------------------------+-----------------------+-----------------+---------------+-----------+-----------------+------------------+------------------+---------------------+-----

###Xử lý

In [27]:
# Tạo Version 2: HANDLE MISSING DATA
# # Tạo bản mới từ Version 1
players_v2 = players_v1
teams_v2 = teams_v1

Xoá dòng thiếu dữ liệu

Chỉ xoá những dòng thiếu ID (critical columns)

In [38]:
from pyspark.sql.functions import col

players_v2 = players_v2.filter(col("player_id").isNotNull())
teams_v2 = teams_v2.filter(col("team_id").isNotNull())

ÉP KIỂU NUMERIC

In [40]:
from pyspark.sql.types import NumericType

# Ép toàn bộ cột numeric về double
for c, dtype in players_v2.dtypes:
    if dtype in ["int", "bigint", "float", "double"]:
        players_v2 = players_v2.withColumn(c, col(c).cast("double"))

for c, dtype in teams_v2.dtypes:
    if dtype in ["int", "bigint", "float", "double"]:
        teams_v2 = teams_v2.withColumn(c, col(c).cast("double"))

Fill NUMERIC bằng MEAN

In [41]:
from pyspark.sql.functions import mean

# ===== PLAYERS =====
numeric_cols_players = [c for c, dtype in players_v2.dtypes
                       if dtype == "double"]

mean_values_players = players_v2.select([
    mean(col(c)).alias(c) for c in numeric_cols_players
]).collect()[0].asDict()

# loại bỏ cột có mean = None
mean_values_players = {k: v for k, v in mean_values_players.items() if v is not None}

players_v2 = players_v2.fillna(mean_values_players)


# ===== TEAMS =====
numeric_cols_teams = [c for c, dtype in teams_v2.dtypes
                     if dtype == "double"]

mean_values_teams = teams_v2.select([
    mean(col(c)).alias(c) for c in numeric_cols_teams
]).collect()[0].asDict()

mean_values_teams = {k: v for k, v in mean_values_teams.items() if v is not None}

teams_v2 = teams_v2.fillna(mean_values_teams)

PHẦN 5: Fill STRING bằng "Unknown"

In [42]:
# ===== PLAYERS =====
string_cols_players = [c for c, dtype in players_v2.dtypes if dtype == "string"]
players_v2 = players_v2.fillna({c: "Unknown" for c in string_cols_players})


# ===== TEAMS =====
string_cols_teams = [c for c, dtype in teams_v2.dtypes if dtype == "string"]
teams_v2 = teams_v2.fillna({c: "Unknown" for c in string_cols_teams})

In [43]:
# Kiểm tra lại sau khi xử lý
check_missing(players_v2, "male_players_v2")
check_missing(teams_v2, "male_teams_v2")


===== MISSING VALUES: male_players_v2 =====
+---------+----------+------------+-----------+----------------+----------+---------+----------------+-------+---------+---------+--------+---+---+---------+---------+---------+-----------+------------+------------+---------+-------------+------------------+----------------+----------------+------------------------------+--------------+----------------+--------------+---------------+--------------------+--------------+---------+-----------+------------------------+---------+---------+---------+------------------+-----------+-------------+----+--------+-------+---------+---------+------+------------------+-------------------+--------------------------+-----------------------+-----------------+---------------+-----------+-----------------+------------------+------------------+---------------------+---------------------+----------------+------------------+----------------+----------------+-------------+-------------+--------------+-------------

In [45]:
players_v2.show(5)

+---------+--------------------+------------+-----------+----------------+--------------+--------------------+----------------+-------+---------+---------+--------+----+----------+---------+---------+---------+--------------+------------+------------+-------------------+-------------+------------------+----------------+----------------+------------------------------+--------------+----------------+--------------+---------------+--------------------+--------------+---------+-----------+------------------------+-------------+----------------+---------+------------------+--------------------+--------------------+----+--------+-------+---------+---------+------+------------------+-------------------+--------------------------+-----------------------+-----------------+---------------+-----------+-----------------+------------------+------------------+---------------------+---------------------+----------------+------------------+----------------+----------------+-------------+-------------+

###Xoá dữ liệu trùng (duplicate)

In [46]:
# Tạo Version 3: REMOVE DUPLICATES
players_v3 = players_v2
teams_v3 = teams_v2


In [47]:
# Đếm số dòng trước khi xoá

# Đếm số dòng ban đầu
players_before = players_v3.count()
teams_before = teams_v3.count()

print("Players BEFORE:", players_before)
print("Teams BEFORE:", teams_before)

Players BEFORE: 10003590
Teams BEFORE: 385055


Xoá duplicate theo ID

Vì:

player_id là khóa chính

team_id là khóa chính

In [48]:
# Xoá duplicate theo ID
players_v3 = players_v3.dropDuplicates(["player_id"])
teams_v3 = teams_v3.dropDuplicates(["team_id"])

In [49]:
# Đếm lại sau khi xoá
players_after = players_v3.count()
teams_after = teams_v3.count()

print("Players AFTER:", players_after)
print("Players REMOVED:", players_before - players_after)

print("Teams AFTER:", teams_after)
print("Teams REMOVED:", teams_before - teams_after)

Players AFTER: 56880
Players REMOVED: 9946710
Teams AFTER: 1112
Teams REMOVED: 383943


In [50]:
# Kiểm tra duplicate còn không (optional)
from pyspark.sql.functions import count

# Kiểm tra player_id bị trùng
players_v3.groupBy("player_id").count().filter("count > 1").show()

# Kiểm tra team_id bị trùng
teams_v3.groupBy("team_id").count().filter("count > 1").show()

+---------+-----+
|player_id|count|
+---------+-----+
+---------+-----+

+-------+-----+
|team_id|count|
+-------+-----+
+-------+-----+



###Loại bỏ dữ liệu sai:



In [66]:
# Tạo Version 4: REMOVE INVALID DATA
players_v4 = players_v3
teams_v4 = teams_v3

Xác định cột numeric quan trọng

FIFA dataset → các cột numeric cần kiểm tra:

age

height_cm

weight_kg

value_eur

wage_eur

overall, potential

skill stats (0–100)

In [67]:
# Loại bỏ giá trị âm bất hợp lý
from pyspark.sql.functions import col

# ===== PLAYERS =====
players_v4 = players_v4.filter(
    (col("age") >= 0) &
    (col("height_cm") > 0) &
    (col("weight_kg") > 0) &
    (col("value_eur") >= 0) &
    (col("wage_eur") >= 0) &
    (col("overall") >= 0) &
    (col("potential") >= 0)
)

# ===== TEAMS =====
teams_v4 = teams_v4.filter(
    (col("overall") >= 0) &
    (col("attack") >= 0) &
    (col("midfield") >= 0) &
    (col("defence") >= 0)
)

Loại bỏ dữ liệu vượt ngưỡng (out-of-range)

FIFA có range rõ ràng:

overall: 0–100

skill: 0–100

age: 15–60 (hợp lý)

In [68]:
# ===== PLAYERS =====
players_v4 = players_v4.filter(
    (col("age").between(15, 60)) &
    (col("overall").between(0, 100)) &
    (col("potential").between(0, 100))
)

# ===== TEAMS =====
teams_v4 = teams_v4.filter(
    (col("overall").between(0, 100)) &
    (col("attack").between(0, 100)) &
    (col("midfield").between(0, 100)) &
    (col("defence").between(0, 100))
)

Xử lý lỗi format (quan trọng)

Một số cột dạng:

"87+3" (rating FIFA)

string thay vì số

Cần convert về số

In [69]:
from pyspark.sql.functions import regexp_replace

# Ví dụ xử lý các cột dạng "87+3" → 87
rating_cols = ["ls","st","rs","lw","lf","cf","rf","rw"]

for c in rating_cols:
    players_v4 = players_v4.withColumn(
        c,
        regexp_replace(col(c), r"\+.*", "").cast("double")
    )

Kiểm tra lại số dòng

In [70]:
# Đếm lại sau khi clean
print("Players:", players_v4.count())
print("Teams:", teams_v4.count())

Players: 56880
Teams: 1112


###Chuẩn hoá kiểu dữ liệu:

In [71]:
# Tạo Version 5: DATA TYPE NORMALIZATION
players_v5 = players_v4
teams_v5 = teams_v4

STRING → NUMBER

Chỉ convert các cột đáng lẽ phải là số

In [72]:
# Players
from pyspark.sql.functions import col

players_v5 = players_v5 \
    .withColumn("age", col("age").cast("int")) \
    .withColumn("height_cm", col("height_cm").cast("float")) \
    .withColumn("weight_kg", col("weight_kg").cast("float")) \
    .withColumn("overall", col("overall").cast("int")) \
    .withColumn("potential", col("potential").cast("int")) \
    .withColumn("value_eur", col("value_eur").cast("float")) \
    .withColumn("wage_eur", col("wage_eur").cast("float"))

In [73]:
# Teams
teams_v5 = teams_v5 \
    .withColumn("overall", col("overall").cast("int")) \
    .withColumn("attack", col("attack").cast("int")) \
    .withColumn("midfield", col("midfield").cast("int")) \
    .withColumn("defence", col("defence").cast("int")) \
    .withColumn("transfer_budget_eur", col("transfer_budget_eur").cast("float")) \
    .withColumn("club_worth_eur", col("club_worth_eur").cast("float"))

DATE → DATETIME

Dùng to_date() hoặc to_timestamp()

In [74]:
# Players
from pyspark.sql.functions import to_date

players_v5 = players_v5 \
    .withColumn("dob", to_date(col("dob"), "yyyy-MM-dd")) \
    .withColumn("fifa_update_date", to_date(col("fifa_update_date"), "yyyy-MM-dd")) \
    .withColumn("club_joined_date", to_date(col("club_joined_date"), "yyyy-MM-dd"))

In [75]:
# Teams
teams_v5 = teams_v5 \
    .withColumn("fifa_update_date", to_date(col("fifa_update_date"), "yyyy-MM-dd"))

KIỂM TRA KẾT QUẢ

In [76]:
print("===== PLAYERS SCHEMA =====")
players_v5.printSchema()

print("===== TEAMS SCHEMA =====")
teams_v5.printSchema()

===== PLAYERS SCHEMA =====
root
 |-- player_id: double (nullable = false)
 |-- player_url: string (nullable = false)
 |-- fifa_version: double (nullable = false)
 |-- fifa_update: double (nullable = false)
 |-- fifa_update_date: date (nullable = true)
 |-- short_name: string (nullable = false)
 |-- long_name: string (nullable = false)
 |-- player_positions: string (nullable = false)
 |-- overall: integer (nullable = true)
 |-- potential: integer (nullable = true)
 |-- value_eur: float (nullable = false)
 |-- wage_eur: float (nullable = false)
 |-- age: integer (nullable = true)
 |-- dob: date (nullable = true)
 |-- height_cm: float (nullable = false)
 |-- weight_kg: float (nullable = false)
 |-- league_id: double (nullable = false)
 |-- league_name: string (nullable = false)
 |-- league_level: double (nullable = false)
 |-- club_team_id: double (nullable = false)
 |-- club_name: string (nullable = false)
 |-- club_position: string (nullable = false)
 |-- club_jersey_number: double (nul